# Cell tracking with the customized Win10 3DeeCellTracker workflow

This notebook follows the laboratory's Win10 `single_mode_worm1-clear.ipynb`:
initial U-Net segmentation, manual correction of the first 3D volume, per-recording
U-Net fine-tuning, then single-mode FFN + PR-GLS tracking.

Install the pinned upstream source and replace the four files as described in
[README.md](README.md), then restart the kernel. Run the cells in order, pausing
for manual correction. Use classic Jupyter Notebook for `%matplotlib notebook`.

Method: [Wen et al., eLife 2021, 10:e59187](https://doi.org/10.7554/eLife.59187).
Original code: [WenChentao/3DeeCellTracker](https://github.com/WenChentao/3DeeCellTracker).
Historical adaptation: [vislabwwy/3DCT_for_PKU_LDG](https://github.com/vislabwwy/3DCT_for_PKU_LDG).
Third-party code retains its [MIT license](LICENSE_3DeeCellTracker).


## 1. Configure one recording

The defaults below come from the Win10 256-mode notebook and are examples, not
settings for every recording. Set the volume count, Z depth, image naming pattern,
resolution ratio and segmentation/tracking parameters for your images.

`MODEL_PROFILE` selects the complete XY field size. The U-Net input patches are
80 x 80 x 5 (128 mode) and 80 x 80 x 16 (256 mode). Both profiles use the same
three-neighbor FFN. `MIN_SIZE` is a region-size threshold in voxels.


In [ ]:
from pathlib import Path

BUNDLE_DIR = Path.cwd().resolve()  # Folder containing this notebook.
PROJECT_DIR = BUNDLE_DIR / 'work' / 'experiment_01'  # One folder per recording.
MODEL_PROFILE = '256x256'  # '128x128' or '256x256'
VOLUME_NUM = 135
Z_SLICES = 46
IMAGE_NAME = 'Den_t%03i_z%03i.tif'  # Time and Z numbering both start at 1.
Z_XY_RATIO = 4.98
Z_SCALING = 5
NOISE_LEVEL = 20
MIN_SIZE = 100
INITIAL_BETA = 300
INITIAL_LAMBDA = 0.1
INITIAL_MAXITER = 20
TRACKING_BETA = 300
TRACKING_LAMBDA = 0.2
TRACKING_MAXITER = 500
RETRAIN_ITERATIONS = 10
MATCH_VOLUME = 7
RANDOM_SEED = None  # Original notebook did not fix a seed; optionally set an int.


## 2. Verify the replacement files and import the tracker

This check prevents an unmodified or different `CellTracker` installation from
being used accidentally. The replacement files are exact copies of the Win10
sources. Restart the kernel whenever you replace installed source files.


In [ ]:
import hashlib
import importlib.util
import json
import shutil
import sys

UPSTREAM_COMMIT = '6819c761d586a1f66cd5615bcfd7fa9bcf5992f8'
REPLACEMENT_SHA256 = {'tracker.py': '573ebd8aa4212cebf24bd5a3157df3df8b7d3d35ecce710d37e42b905b57f6c7', 'unet3d.py': 'b65b28ee4a29fb019eafdb6096a1d4fd66d22437aa2c9e1fbf10222dc3ec445a', 'ffn.py': 'e3d24222e97bc9f2849f7f4cc7d89d24ba686103121fd0f5e1d89df24d26f873', 'synthesize.py': '732559ecb5aa5cc3b51933c6ceeca0607d770e967d656a5442fc2e0ba215d3f3'}
MODEL_SHA256 = {'128x128/unet3_pretrained.h5': 'e12f7f1e3d56c1a4cb8dad6ea9f53464fe0d7e508a51f1e85675a265e143f789', '256x256/unet3_pretrained.h5': '87cce6ec2d22b38a4a321ec8b811bca02b33145c16aecac091c98f74825b2a1b', 'ffn_k3.hdf5': '3b455a2d528f21db0f07cb18d2af1b259ea2bfac3a54be80acb1281a3162210a'}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

assert (BUNDLE_DIR / 'Cell_tracking.ipynb').is_file(), (
    'Set BUNDLE_DIR to the folder containing Cell_tracking.ipynb.')
assert MODEL_PROFILE in ('128x128', '256x256'), 'Choose one supported model profile.'
assert VOLUME_NUM >= 2 and Z_SLICES >= 1
assert 1 < MATCH_VOLUME <= VOLUME_NUM, 'Set MATCH_VOLUME within this recording.'
spec = importlib.util.find_spec('CellTracker')
if spec is None or not spec.submodule_search_locations:
    raise ImportError('Install the patched upstream source using README.md first.')
installed_package = Path(next(iter(spec.submodule_search_locations))).resolve()
for name, expected in REPLACEMENT_SHA256.items():
    supplied = BUNDLE_DIR / 'replacement_files' / 'CellTracker' / name
    installed = installed_package / name
    if not supplied.is_file() or sha256_file(supplied) != expected:
        raise RuntimeError('The supplied replacement file changed: ' + name)
    if not installed.is_file() or sha256_file(installed) != expected:
        raise RuntimeError('Replace ' + str(installed) + ' using README.md, then restart the kernel.')
print('Using patched package:', installed_package)


In [ ]:
%matplotlib inline
import numpy as np
import tensorflow as tf
import h5py
from tifffile import imread
from IPython.display import display, HTML
from CellTracker.tracker import Tracker

if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)
    tf.random.set_seed(RANDOM_SEED)
print('Python:', sys.version.split()[0], '| TensorFlow:', tf.__version__)


## 3. Initialize the recording and copy the selected models

The bundled initial models are copied into `PROJECT_DIR/models`. Existing files
are reused only when they have the expected checksum. Use a separate project
folder for every recording and model profile.


In [ ]:
xy_size = {'128x128': 128, '256x256': 256}[MODEL_PROFILE]
expected_patch = {'128x128': (80, 80, 5), '256x256': (80, 80, 16)}[MODEL_PROFILE]
project_models = PROJECT_DIR / 'models'
project_models.mkdir(parents=True, exist_ok=True)
selected_assets = {
    MODEL_PROFILE + '/unet3_pretrained.h5': 'unet3_pretrained.h5',
    'ffn_k3.hdf5': 'ffn_k3.hdf5',
}
for relative, filename in selected_assets.items():
    src = BUNDLE_DIR / 'models' / relative
    dst = project_models / filename
    expected = MODEL_SHA256[relative]
    if not src.is_file() or sha256_file(src) != expected:
        raise RuntimeError('Missing or changed bundled model: ' + str(src))
    if dst.exists() and sha256_file(dst) != expected:
        raise RuntimeError('Different initial model already exists at ' + str(dst) +
                           '. Select a separate PROJECT_DIR.')
    if not dst.exists():
        shutil.copy2(src, dst)

tracker = Tracker(
    volume_num=VOLUME_NUM, siz_xyz=(xy_size, xy_size, Z_SLICES),
    z_xy_ratio=Z_XY_RATIO, z_scaling=Z_SCALING,
    noise_level=NOISE_LEVEL, min_size=MIN_SIZE,
    beta_tk=INITIAL_BETA, lambda_tk=INITIAL_LAMBDA, maxiter_tk=INITIAL_MAXITER,
    folder_path=str(PROJECT_DIR), image_name=IMAGE_NAME,
    unet_model_file='unet3_pretrained.h5', ffn_model_file='ffn_k3.hdf5')

run_info = {
    'upstream_commit': UPSTREAM_COMMIT,
    'replacement_sha256': REPLACEMENT_SHA256,
    'model_profile': MODEL_PROFILE, 'initial_model_sha256': {
        key: MODEL_SHA256[key] for key in selected_assets},
    'project_dir': str(PROJECT_DIR), 'image_name': IMAGE_NAME,
    'volume_num': VOLUME_NUM, 'siz_xyz': [xy_size, xy_size, Z_SLICES],
    'z_xy_ratio': Z_XY_RATIO, 'z_scaling': Z_SCALING,
    'noise_level': NOISE_LEVEL, 'min_size': MIN_SIZE,
    'initial_tracking': [INITIAL_BETA, INITIAL_LAMBDA, INITIAL_MAXITER],
    'tracking': [TRACKING_BETA, TRACKING_LAMBDA, TRACKING_MAXITER],
    'retrain_iterations': RETRAIN_ITERATIONS, 'random_seed': RANDOM_SEED,
    'python': sys.version, 'tensorflow': tf.__version__, 'numpy': np.__version__,
    'match_volume': MATCH_VOLUME, 'stage': 'initialized',
}

def save_run_info():
    (PROJECT_DIR / 'run_config.json').write_text(
        json.dumps(run_info, indent=2), encoding='utf-8')

save_run_info()
print('Place the TIFF slices in:', PROJECT_DIR / 'data')


## 4. Prepare images and segment the first volume

Copy the images into `PROJECT_DIR/data`. Each file contains one 2D Z slice.
For the example pattern, names start with `Den_t001_z001.tif`. Preserve the
recording's intensity scale and make sure the configured dimensions match it.


In [ ]:
missing = []
for t in range(1, VOLUME_NUM + 1):
    for z in range(1, Z_SLICES + 1):
        path = PROJECT_DIR / 'data' / (IMAGE_NAME % (t, z))
        if not path.is_file():
            missing.append(str(path))
if missing:
    raise FileNotFoundError('Missing ' + str(len(missing)) +
                            ' image slices. First missing file: ' + missing[0])
first_slice = imread(PROJECT_DIR / 'data' / (IMAGE_NAME % (1, 1)))
assert first_slice.shape == (xy_size, xy_size), (
    'Image XY dimensions do not match MODEL_PROFILE: ' + str(first_slice.shape))

tracker.set_segmentation(noise_level=NOISE_LEVEL, min_size=MIN_SIZE)
tracker.load_unet()
assert tuple(tracker.unet_model.input_shape[1:4]) == expected_patch
tracker.segment_vol1()
anim_seg = tracker.draw_segresult(percentile_high=99.8)
display(HTML(anim_seg))


## 5. Manually correct the first volume, then load its labels

**Pause here.** Correct the first-volume segmentation in an external tool such
as ITK-SNAP: remove false positives, add missed cells and correct boundaries.
Export the completed labels into `PROJECT_DIR/manual_vol1` as one 2D TIFF per
Z slice in lexicographic Z order (for example `label_z001.tif`, `label_z002.tif`).

Use 0 for background and a distinct positive integer for each cell, with the
same cell ID across its Z slices. Export label values, not RGB screenshots.
The folder must contain exactly the Z slices of volume 1, with no extra files.
Both the reference and subsequently detected point sets must contain at least
four cells for the three-neighbor FFN.


In [ ]:
manual_dir = PROJECT_DIR / 'manual_vol1'
manual_files = sorted(manual_dir.iterdir())
if len(manual_files) != Z_SLICES or any(
        not p.is_file() or p.suffix.lower() not in ('.tif', '.tiff') for p in manual_files):
    raise ValueError('manual_vol1 must contain exactly Z_SLICES label TIFFs and no extra files.')
manual_planes = [imread(p) for p in manual_files]
if any(a.shape != (xy_size, xy_size) for a in manual_planes):
    raise ValueError('Manual-label XY dimensions must match the imaging slices.')
labels = np.stack(manual_planes, axis=2)
if not np.all(np.isfinite(labels)) or np.any(labels < 0) or np.any(labels != np.floor(labels)):
    raise ValueError('Manual segmentation must contain nonnegative integer labels.')
cell_ids = np.unique(labels[labels > 0])
if len(cell_ids) < 4:
    raise ValueError('The manual reference needs at least four cells for k=3.')
run_info['manual_label_files'] = [
    {'name': p.name, 'sha256': sha256_file(p)} for p in manual_files]
run_info['manual_cell_count'] = int(len(cell_ids))
tracker.load_manual_seg()
run_info['stage'] = 'manual_labels_loaded'
save_run_info()
print('Manual cells:', len(cell_ids))


## 6. Fine-tune U-Net on this recording's first volume

The original workflow trains for up to 10 steps and saves a checkpoint only
when validation loss improves. Review the predictions and choose a saved step
in the next cell. Training is recording-specific; bundled weights are the
initial models. Keep the manual labels and the final selected checkpoint.


In [ ]:
if list((project_models / 'unet_weights').glob('unet_weights_retrain_step*.h5')):
    raise RuntimeError('Fine-tuning checkpoints already exist. Use a fresh project folder for new training, '
                       'or skip this training cell to select an archived checkpoint explicitly.')
tracker.retrain_unet(iteration=RETRAIN_ITERATIONS)
print('Saved checkpoints:')
for path in sorted((project_models / 'unet_weights').glob('unet_weights_retrain_step*.h5')):
    print(path.name)


In [ ]:
# Choose a checkpoint saved for this recording after reviewing the predictions.
SELECTED_UNET_STEP = 9  # Historical Win10 example; 0 selects the initial model.
if not 0 <= SELECTED_UNET_STEP <= RETRAIN_ITERATIONS:
    raise ValueError('Select a step between 0 and RETRAIN_ITERATIONS.')
checkpoint_name = ('weights_initial.h5' if SELECTED_UNET_STEP == 0 else
                   'unet_weights_retrain_step' + str(SELECTED_UNET_STEP) + '.h5')
checkpoint = project_models / 'unet_weights' / checkpoint_name
if not checkpoint.is_file():
    raise FileNotFoundError('This step was not saved: ' + str(checkpoint) +
                            '. Choose an available checkpoint.')
tracker.select_unet_weights(step=SELECTED_UNET_STEP)
tracker.set_segmentation(del_cache=True)
tracker.segment_vol1()
anim_seg = tracker.draw_segresult(percentile_high=99.8)
run_info['selected_unet_step'] = SELECTED_UNET_STEP
run_info['selected_checkpoint_sha256'] = sha256_file(checkpoint)
run_info['stage'] = 'unet_checkpoint_selected'
save_run_info()
display(HTML(anim_seg))


## 7. Initialize and inspect tracking

The bundled FFN has two 10-feature inputs, consistent with k=3. The initial
manual labels are interpolated as in the Win10 source notebook. Check the
test match and correction before starting all subsequent volumes.


In [ ]:
tracker.interpolate_seg()
tracker.draw_manual_seg1()
tracker.cal_subregions()
tracker.load_ffn()
ffn_shapes = tracker.ffn_model.input_shape
assert len(ffn_shapes) == 2 and all(shape[-1] == 10 for shape in ffn_shapes)
tracker.initiate_tracking()
tracker.set_tracking(beta_tk=TRACKING_BETA, lambda_tk=TRACKING_LAMBDA,
                     maxiter_tk=TRACKING_MAXITER)


In [ ]:
anim_tracking, match_results = tracker.match(target_volume=MATCH_VOLUME)
display(HTML(anim_tracking))


In [ ]:
tracker.draw_correction(*match_results[2:])
tracker.draw_overlapping(*match_results[:3])


## 8. Track subsequent volumes and review the output

Use the classic Notebook interactive backend for the original tracking display.
If a frame produces fewer than four detected cells, inspect that frame and
its segmentation parameters; changing the neighbor count alone would also
require a matching FFN model. Keep the parameters and selected checkpoint
with the completed tracking results.


In [ ]:
%matplotlib notebook
fig, ax = tracker.subplots_tracking()
tracker.track(fig, ax, from_volume=2)
run_info['stage'] = 'tracking_complete'
save_run_info()


In [ ]:
%matplotlib inline
track_anim = tracker.replay_track_animation()
display(HTML(track_anim))


Outputs follow the upstream layout under `PROJECT_DIR`:

- `track_results_SingleMode/`: tracked label slices.
- `track_information/`: tracking coordinates/information.
- `anim/`: tracking process images.
- `models/unet_weights/`: recording-specific checkpoints and selected model.
- `run_config.json`: effective parameters, source/model hashes and selected step.

`auto_vol1/` and `unet_cache/` are intermediate segmentation outputs. Preserve
the images, manual labels, selected checkpoint and run configuration needed
to reproduce the result. Use a fresh project folder when changing records.
